# 12주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 과제 내용
직접 문장 10개를 작성해 모델의 예측을 확인하고, 틀린 예측의 원인을 추정합니다.

## 제출 방법
모든 셀을 실행해 출력과 표가 보이는 상태로 저장한 뒤 `week12_학번_이름.ipynb` 로 제출합니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re, random, os, urllib.request, tarfile
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

torch.manual_seed(42); random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# 고정 조건 — 바꾸지 마세요 (문제 3에서만 변경)
PAD, UNK = 0, 1
MAX_LEN, MAX_VOCAB, EPOCHS, BATCH, LR, SEED = 200, 10000, 6, 64, 1e-3, 42


## 데이터와 공통 함수
아래를 그대로 쓰세요.


In [ ]:
URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
if not os.path.exists("aclImdb"):
    print("내려받는 중...")
    urllib.request.urlretrieve(URL, "aclImdb.tar.gz")
    with tarfile.open("aclImdb.tar.gz") as f:
        f.extractall(".")

def tokenize(s):
    s = s.lower().replace("<br />", " ")
    return re.sub(r"[^a-z0-9'\s]", " ", s).split()

def load_split(split, n=2500):
    data = []
    for label, name in [(1, "pos"), (0, "neg")]:
        d = f"aclImdb/{split}/{name}"
        for fn in sorted(os.listdir(d))[:n]:
            with open(os.path.join(d, fn), encoding="utf-8") as f:
                data.append((f.read(), label))
    random.shuffle(data)
    return data


train_data, test_data = load_split("train"), load_split("test", 1250)
train_tokens = [tokenize(t) for t, _ in train_data]
test_tokens  = [tokenize(t) for t, _ in test_data]
LABELS = ["부정", "긍정"]


def build_vocab(token_lists, max_vocab=MAX_VOCAB):
    counter = Counter(w for t in token_lists for w in t)
    v = {"<PAD>": PAD, "<UNK>": UNK}
    for w, _ in counter.most_common(max_vocab - 2):
        v[w] = len(v)
    return v


class ReviewDataset(Dataset):
    def __init__(self, tl, labels, vocab, max_len=MAX_LEN):
        seqs = []
        for t in tl:
            ids = [vocab.get(w, UNK) for w in t][:max_len]
            seqs.append(ids + [PAD] * (max_len - len(ids)))
        self.x = torch.tensor(seqs); self.y = torch.tensor(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]


def make_loaders(vocab, max_len=MAX_LEN):
    tr = DataLoader(ReviewDataset(train_tokens, [l for _, l in train_data], vocab, max_len),
                    batch_size=BATCH, shuffle=True)
    te = DataLoader(ReviewDataset(test_tokens, [l for _, l in test_data], vocab, max_len),
                    batch_size=128, shuffle=False)
    return tr, te


loss_fn = nn.CrossEntropyLoss()

def run(model, train_loader, test_loader, epochs=EPOCHS, lr=LR, seed=SEED):
    torch.manual_seed(seed)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    hist = []
    for epoch in range(1, epochs+1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for x, y in test_loader:
                preds.append(model(x.to(device)).argmax(dim=1).cpu()); trues.append(y)
        p, t = torch.cat(preds), torch.cat(trues)
        hist.append((p == t).float().mean().item())
        print(f"  epoch {epoch}  검증 {hist[-1]:.4f}")
    return {"model": model, "hist": hist, "acc": hist[-1],
            "time": time.time()-start,
            "params": sum(q.numel() for q in model.parameters())}

print("준비 완료")


## 문제 1. 감성 분류 모델 작성과 학습 (25점)
`Embedding → LSTM → Linear` 구조의 모델을 작성해 학습하세요.

- `padding_idx` 를 지정하세요
- 임베딩 파라미터가 전체의 몇 %인지 출력하세요
- epoch별 검증 정확도와 학습 곡선을 그리세요


In [ ]:
# 답안


## 문제 2. 직접 쓴 문장 10개 예측 (30점)
자기가 만든 영어 문장 **10개**로 예측을 확인하세요. 아래 조건을 지키세요.

- 긍정 5개, 부정 5개
- 그중 최소 3개는 **까다로운 문장** (부정 표현, 긍정·부정 혼합, 아주 짧은 문장 등)

결과를 표로 출력하세요.

| 문장 | 정답 | 예측 | 확률 | 단어 수 | UNK 수 | 맞음 |


In [ ]:
# 답안


## 문제 3. 틀린 예측의 원인 추정 (25점)
문제 2에서 틀린 문장을 하나씩 짚어 원인을 추정하세요. 아래 근거를 **숫자로** 제시하세요.

- 문장의 단어 수
- UNK 개수와 비율 (어떤 단어가 UNK가 됐는지도 적으세요)
- 예측 확률 (모델이 확신했는지)
- 부정·역접 표현이 있는지

전부 맞혔다면, 더 까다로운 문장 3개를 추가해 틀리는 경우를 찾으세요.


In [ ]:
# 답안


**원인 분석:**

1.

2.

3.


## 문제 4. 설정을 바꿔 개선 시도 (20점)
아래에서 **두 가지 이상**을 골라 바꿔 보고, 문제 2의 문장 10개 정확도가 어떻게 변하는지 비교하세요.

- `MAX_LEN` 변경 (100 / 400)
- `MAX_VOCAB` 변경 (3000 / 20000)
- 양방향 LSTM (`bidirectional=True`)
- 임베딩 차원 변경

| 설정 | 검증 정확도 | 내 문장 10개 정확도 | 파라미터 | 시간 |

어떤 설정이 내 문장에 가장 도움이 됐는지, 그 이유를 두세 줄로 적으세요.


In [ ]:
# 답안


**답:**
